# Data control panel

Local control panel for checking data freshness, updating the tournament master file, and running the project smoke check.

In [ ]:
from __future__ import annotations

import html as html_module
import os
import subprocess
import sys
import threading
from pathlib import Path

from IPython.display import Markdown, display
import ipywidgets as widgets


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Could not find project root.')


def resolve_project_python(project_root: Path) -> str:
    candidates = [
        project_root / '.venv' / 'Scripts' / 'python.exe',
        project_root / 'venv' / 'Scripts' / 'python.exe',
        project_root / '.venv' / 'bin' / 'python',
        project_root / 'venv' / 'bin' / 'python',
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return sys.executable


PROJECT_ROOT = find_project_root()
PYTHON = resolve_project_python(PROJECT_ROOT)
BUTTONS: list[widgets.Button] = []
LOG_LINES: list[str] = []
MAX_LOG_LINES = 1000


def set_buttons_disabled(disabled: bool) -> None:
    for button in BUTTONS:
        button.disabled = disabled


def set_log(text: str) -> None:
    log_area.value = text


def append_log(text: str) -> None:
    for line in text.splitlines() or ['']:
        LOG_LINES.append(line)
    del LOG_LINES[:-MAX_LOG_LINES]
    log_area.value = '\n'.join(LOG_LINES)


def describe_output_line(line: str, title: str) -> str | None:
    text = line.strip()
    if not text:
        return None
    if text.startswith('$ '):
        return 'Launching command'
    if text.startswith('===') and text.endswith('==='):
        return text.strip('= ').strip()
    if text.startswith('Full RTT predictor pipeline'):
        return 'Full pipeline started'
    if text.startswith('Preflight dependency check'):
        return 'Checking Python dependencies'
    if text.startswith('Missing Python packages:'):
        return text
    if text.startswith('Python executable:'):
        return text
    if text.startswith('Project root:'):
        return 'Checking project location'
    if text.startswith('Started at:'):
        return 'Preparing pipeline steps'
    if text.startswith('Parsing calendar age group:'):
        return 'Calendar filter: ' + text.split(':', 1)[1].strip()
    if text.startswith('Fetching tournament details'):
        return text
    if text.startswith('Tournament download plan:'):
        return text
    if text.startswith('future_start_date') or text.startswith('completed_cached') or text.startswith('cancelled'):
        return 'Skipping/caching: ' + text
    if text.startswith('Updated matches_page_saved'):
        return text
    if text.startswith('Parsed tournaments:'):
        return text
    if text.startswith('Master before:') or text.startswith('Master after:'):
        return text
    if text.startswith('New tournaments:'):
        return text
    if text.startswith('Output:'):
        return 'Writing output file: ' + text.split(':', 1)[1].strip()
    if text.startswith('Data status'):
        return 'Reading local data status'
    if text.startswith('Rankings rows:'):
        return 'Checking rankings dataset'
    if text.startswith('Matches rows:'):
        return 'Checking parsed matches dataset'
    if text.startswith('Dataset rows:'):
        return 'Checking model dataset'
    if text.startswith('Wrote') and 'data_manifest' in text:
        return 'Updating data manifest'
    if text.startswith('notebook ok:'):
        return 'Verifying notebook: ' + text.split(':', 1)[1].strip()
    if text.startswith('dataset ok'):
        return 'Verifying final dataset'
    if text.startswith('tournaments ok'):
        return 'Verifying tournament master'
    if text.startswith('project verification passed'):
        return 'Project verification passed'
    if text.startswith('Step failed:'):
        return text
    if 'ModuleNotFoundError' in text:
        return text
    return None


def set_current_step(text: str) -> None:
    current_step_html.value = '<b>Now:</b> ' + html_module.escape(text)


def run_project_command_streaming(title: str, args: list[str]) -> None:
    set_buttons_disabled(True)
    progress.value = 0
    progress.bar_style = 'info'
    status_html.value = f'<b>{title}</b>: running...'
    set_current_step('Starting ' + title)

    command = [PYTHON, '-u', *args]
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['PYTHONUTF8'] = '1'

    LOG_LINES.clear()
    set_log('')
    append_log(f'### {title}')
    append_log('$ ' + ' '.join(command))
    append_log('')

    try:
        process = subprocess.Popen(
            command,
            cwd=PROJECT_ROOT,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )

        line_count = 0
        assert process.stdout is not None
        for line in process.stdout:
            line_count += 1
            progress.value = min(99, 5 + line_count)
            current_description = describe_output_line(line, title)
            if current_description:
                set_current_step(current_description)
            append_log(line.rstrip('\n'))

        return_code = process.wait()
        if return_code == 0:
            progress.value = 100
            progress.bar_style = 'success'
            status_html.value = f'<b>{title}</b>: done'
            set_current_step('Completed ' + title)
        else:
            progress.value = 100
            progress.bar_style = 'danger'
            status_html.value = f'<b>{title}</b>: failed with code {return_code}'
            set_current_step(f'Stopped after error code {return_code}')
    except Exception as exc:
        progress.value = 100
        progress.bar_style = 'danger'
        status_html.value = f'<b>{title}</b>: failed'
        set_current_step('Stopped after an unexpected error')
        append_log(str(exc))
    finally:
        set_buttons_disabled(False)


def show_command(title: str, args: list[str]) -> None:
    thread = threading.Thread(target=run_project_command_streaming, args=(title, args), daemon=True)
    thread.start()


status_button = widgets.Button(description='Refresh status', button_style='info')
tournaments_button = widgets.Button(description='Update tournaments master', button_style='warning')
calendar_button = widgets.Button(description='Update RTT calendar', button_style='warning')
verify_button = widgets.Button(description='Verify project', button_style='success')
pipeline_button = widgets.Button(description='Run full pipeline', button_style='danger')
BUTTONS.extend([status_button, tournaments_button, calendar_button, pipeline_button, verify_button])

progress = widgets.IntProgress(value=0, min=0, max=100, description='Progress:', bar_style='')
status_html = widgets.HTML(value='Ready')
current_step_html = widgets.HTML(value='<b>Now:</b> idle')
log_area = widgets.Textarea(
    value='',
    disabled=True,
    layout=widgets.Layout(width='100%', height='360px'),
)

status_button.on_click(lambda _: show_command('Data status', ['scripts/data_status.py', '--write-manifest']))
tournaments_button.on_click(lambda _: show_command('Tournament master update', ['scripts/bootstrap_tournaments_master.py']))
calendar_button.on_click(lambda _: show_command('RTT calendar update', ['scripts/parse_rtt_calendar.py']))
pipeline_button.on_click(lambda _: show_command('Full update pipeline', ['scripts/run_full_pipeline.py']))
verify_button.on_click(lambda _: show_command('Project verification', ['scripts/verify_project.py']))

display(Markdown(f'Project root: `{PROJECT_ROOT}`'))
display(Markdown(f'Python: `{PYTHON}`'))
display(widgets.HBox([status_button, tournaments_button, calendar_button, pipeline_button, verify_button]))
display(widgets.VBox([status_html, current_step_html, progress, log_area]))
show_command('Data status', ['scripts/data_status.py', '--write-manifest'])
